# AOI-PCB-SSD: Live Camera Inference

A live webcam demo of the trained detector, reproducing the real-time inspection
setup from Section VII of the paper. Each frame is cropped square, resized to the
model's input size, and overlaid with the predicted IC corner points and class
confidence.

> **Note:** This notebook opens an OpenCV window and must run on a local machine
> with a camera — it cannot run headless (e.g. in CI). Press **ESC** to stop the
> capture loop.

### Prerequisites
1. Install the package: `pip install -e ".[dev]"`
2. Train a model: `python scripts/train.py --architecture custom`

### Contents
1. Setup & load model
2. Detection overlay helper
3. Live webcam loop

## 1. Setup & Load Model

In [ ]:
from pathlib import Path

import cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras.optimizers import Adam

from aoi_pcb_ssd.config_loader import Config
from aoi_pcb_ssd.encoding.output_decoder import decode_detections
from aoi_pcb_ssd.model.grid_centers import GridCenters
from aoi_pcb_ssd.model.loss import AOILoss
from aoi_pcb_ssd.model.metrics import class_mAP, mae

MODEL_PATH = None  # set explicitly, or auto-select the most recent run

if MODEL_PATH is None:
    runs = sorted(
        Path("../experiments").glob("*/model.keras"),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    if not runs:
        raise FileNotFoundError("No trained model in ../experiments/. Run scripts/train.py first.")
    model_path = runs[0]
else:
    model_path = Path(MODEL_PATH)

config = Config(str(model_path.parent / "config.json"))
m = config.model

aoi_loss = AOILoss(**config.get_init_kwargs("training.loss"))
model = tf.keras.models.load_model(
    str(model_path), compile=False, custom_objects={"GridCenters": GridCenters}
)
model.compile(
    optimizer=Adam(**config.get_init_kwargs("training.optimizer")),
    loss=aoi_loss.compute_loss,
    metrics=[class_mAP, mae],
)
print(f"Loaded model: {model_path}")

## 2. Detection Overlay Helper

Draws each predicted IC's four corner circles and a class/confidence label onto
a BGR frame. The model outputs raw predictions; `decode_detections` (called in
the loop) converts them to pixel-space corners first.

In [ ]:
_CLASSES = ["background", "ic"]


def plot_detections(decoded_pred: np.ndarray, image: np.ndarray) -> np.ndarray:
    """Overlay predicted IC corner circles and a class/confidence label on a frame."""
    if decoded_pred.shape[0] == 0:
        return image
    for det in decoded_pred:
        corners = np.reshape(det[2:], (-1, 2)).astype(int)
        for point in corners:
            image = cv2.circle(image, tuple(point), 2, (0, 0, 255), 1)
        center_x = int((corners[0, 0] + corners[3, 0]) / 2)
        center_y = int((corners[0, 1] + corners[3, 1]) / 2)
        label = f"{_CLASSES[int(det[0])]}: {det[1]:.2f}"
        image = cv2.putText(
            image, label, (center_x, center_y),
            cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2, cv2.LINE_AA,
        )
    return image

## 3. Live Webcam Loop

Opens the default camera, runs detection on each frame, and shows the overlaid
result at 2× scale. Press **ESC** to quit; the camera is always released.

In [ ]:
cam = cv2.VideoCapture(0)
try:
    while True:
        ret, frame = cam.read()
        if not ret:
            break
        side = min(frame.shape[:2])
        frame = frame[:side, :side]
        frame = cv2.resize(frame, (m.img_width, m.img_height), interpolation=cv2.INTER_AREA)

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        predictions = model.predict(np.expand_dims(rgb, axis=0), verbose=0)
        decoded = decode_detections(
            predictions,
            normalize_coords=m.normalize_coords,
            img_height=m.img_height,
            img_width=m.img_width,
        )

        frame = plot_detections(decoded[0], frame)
        frame = cv2.resize(frame, (m.img_width * 2, m.img_height * 2), interpolation=cv2.INTER_AREA)
        cv2.imshow("AI AOI", frame)

        if cv2.waitKey(1) == 27:  # ESC
            break
finally:
    cam.release()
    cv2.destroyAllWindows()